In [1]:
import numpy as np
import pandas as pd

from task6.utils.balance_dataset import augment_minority_classes
from task6.utils.prepare_data import prepare_data

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Szymon\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
df = pd.read_csv('../../data/raya.csv')
df.head()

,dialogue_id,utterance_id,sentence_id,topic,dialog_act,emotion,sentence_text,sentence_text_bg
0,0,0,0,Ordinary Life,directive,disgust,The kitchen stinks .,Кухнята мирише.
1,0,1,0,Ordinary Life,commissive,no emotion,I'll throw out the garbage .,Ще изхвърля боклука.
2,1,0,0,Ordinary Life,directive,happiness,"So Dick , how about getting some coffee for to...","Така че Dick , какво ще кажете за получаване н..."
3,1,1,0,Ordinary Life,commissive,disgust,Coffee ?,Кафе ?
4,1,1,1,Ordinary Life,commissive,disgust,I don ’ t honestly like that kind of stuff .,"Честно казано, не харесвам такива неща."


In [3]:
df = df.drop(columns="sentence_text_bg")

In [4]:
df.emotion.value_counts()

emotion
no emotion    142030
happiness      23314
surprise        3843
sadness         2211
anger           2083
disgust          637
fear             450
Name: count, dtype: int64

In [5]:
# rename joy as happiness to match ekman emotions
df['emotion'] = df['emotion'].replace({'no emotion': 'neutral'})

In [6]:
df.emotion.value_counts()

emotion
neutral      142030
happiness     23314
surprise       3843
sadness        2211
anger          2083
disgust         637
fear            450
Name: count, dtype: int64

In [7]:
# undersample neutral and happiness to 5000 each
df_neutral = df[df['emotion'] == 'neutral'].sample(n=5000, random_state=42)
df_happiness = df[df['emotion'] == 'happiness'].sample(n=5000, random_state=42)
df_other = df[df['emotion'].isin(['anger', 'sadness', 'fear', 'disgust', 'surprise'])]
df = pd.concat([df_neutral, df_happiness, df_other]).reset_index(drop=True)
df['emotion'].value_counts()

emotion
neutral      5000
happiness    5000
surprise     3843
sadness      2211
anger        2083
disgust       637
fear          450
Name: count, dtype: int64

In [8]:
df, emotions = prepare_data(df, "sentence_text", "emotion")
print(emotions)
df.head()

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed
['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']


,dialogue_id,utterance_id,sentence_id,topic,dialog_act,sentence_text,ekman_emotion,tokenized_text,lemmatized_text
0,12990,1,1,Finance,directive,I'd like to cash this check .,6,"[i, 'd, like, to, cash, this, check, .]","[I, would, like, to, cash, this, check, .]"
1,11949,8,0,Work,directive,What kind of action is required ?,6,"[what, kind, of, action, is, required, ?]","[what, kind, of, action, be, require, ?]"
2,1631,1,1,Ordinary Life,question,That's not too bad .,6,"[that, 's, not, too, bad, .]","[that, be, not, too, bad, .]"
3,5174,3,1,Relationship,question,I haven't been able to eat a single piece of m...,6,"[i, have, n't, been, able, to, eat, a, single,...","[I, have, not, be, able, to, eat, a, single, p..."
4,5312,12,0,Relationship,directive,"Well , we'd better get back to our seats .",6,"[well, ,, we, 'd, better, get, back, to, our, ...","[well, ,, we, would, well, get, back, to, our,..."


In [9]:
df_test = pd.read_csv("../../data/transcript_spell_checked.csv")
df_test, emotions_test = prepare_data(df_test, "Translation", "emotion_final")

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed


In [10]:
target_samples = 1500

classes_to_augment = df["ekman_emotion"].value_counts()[df["ekman_emotion"].value_counts() < target_samples].index.tolist()
print(f"Classes to augment: {classes_to_augment}")

Classes to augment: [1, 2]


In [11]:
df_augmented = augment_minority_classes(df, target_samples=target_samples, classes_to_augment=classes_to_augment)
df_augmented = pd.concat([df, df_augmented]).reset_index(drop=True)
print(df_augmented["ekman_emotion"].value_counts())
print(f"Original size: {len(df)}, Augmented size: {len(df_augmented)}")
df = df_augmented

Augmenting emotion 1: 637 → 1500
Augmenting emotion 2: 450 → 1500
Augmented Sample 1: you mean that I will not get a raise until bastardly the recession end ?
Augmented Sample 2: so they virus to tell they be smart ?
Augmented Sample 3: I just do not what be wrong .
ekman_emotion
6    5000
3    5000
5    3843
4    2211
0    2083
1    1477
2    1457
Name: count, dtype: int64
Original size: 19224, Augmented size: 21071


In [12]:
df["lemmatized_text"] = df["lemmatized_text"].apply(lambda tokens: " ".join(tokens))
df_test["lemmatized_text"] = df_test["lemmatized_text"].apply(lambda tokens: " ".join(tokens))

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_word = TfidfVectorizer(max_features=40000, ngram_range=(1, 2))
tfidf_word = vectorizer_word.fit_transform(df["lemmatized_text"])
tfidf_word_test = vectorizer_word.transform(df_test["lemmatized_text"])

In [14]:
from transformers import pipeline

# sentiment analysis
sentiment = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment", top_k=None)

C:\Projects\BUAS\Y2\block-a\fae2-nlpr-group-group-9-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0


In [15]:
def analyze_sentiment_batch(texts, max_len=500):
    # Trim all texts upfront
    trimmed_texts = [
        t[:max_len] if isinstance(t, str) and len(t) > max_len else t
        for t in texts
    ]

    # Run through the pipeline in batches
    results = sentiment(trimmed_texts, batch_size=256)

    numeric_scores = []
    for result in results:
        best = max(result, key=lambda x: x["score"])
        label = best["label"]
        conf = best["score"]

        if label == "negative":
            numeric_scores.append(-conf)
        elif label == "neutral":
            numeric_scores.append(0.0)
        elif label == "positive":
            numeric_scores.append(conf)
        else:
            numeric_scores.append(0.0)

    return numeric_scores

In [16]:
df["sentiment_score"] = analyze_sentiment_batch(df["lemmatized_text"].tolist())
df_test["sentiment_score"] = analyze_sentiment_batch(df_test["lemmatized_text"].tolist())

In [17]:
# Intensity markers count: occurrences of "very", "so", "extremely" and other similar words
intensity_markers = ["very", "so", "extremely", "highly", "too", "really", "absolutely", "completely", "totally", "utterly"]
df["intensity_marker_count"] = df["lemmatized_text"].apply(lambda x: sum(x.lower().count(marker) for marker in intensity_markers))
df_test["intensity_marker_count"] = df_test["lemmatized_text"].apply(lambda x: sum(x.lower().count(marker) for marker in intensity_markers))

In [18]:
# load emotion lexicon
emotion_lexicon = pd.read_csv("../../data/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt", sep="\t", names=["word", "emotion", "association"])
# pivot the lexicon to have emotions as columns
emotion_lexicon_pivot = emotion_lexicon.pivot(index="word", columns="emotion", values="association").fillna(0)
# create a dictionary for faster lookup
emotion_dict = emotion_lexicon_pivot.to_dict(orient="index")

In [19]:
def return_emotion_counts(lemmas_str):
    lemmas = lemmas_str.lower().split()
    counts = {emotion: 0 for emotion in emotion_lexicon_pivot.columns}

    for lemma in lemmas:
        if lemma in emotion_dict:
            for emotion in emotion_dict[lemma]:
                # increment only emotions marked 1
                if emotion_dict[lemma][emotion] == 1:
                    counts[emotion] += 1

    return pd.Series(counts)

In [20]:
emotion_counts = df["lemmatized_text"].apply(return_emotion_counts)
df = pd.concat([df, emotion_counts], axis=1)

In [21]:
emotion_counts = df_test["lemmatized_text"].apply(return_emotion_counts)
df_test = pd.concat([df_test, emotion_counts], axis=1)

In [22]:
# calculate number of tokens, ratio of unique tokens to total tokens, average word length
df["num_tokens"] = df["lemmatized_text"].apply(lambda x: len(x.split()))
df["unique_token_ratio"] = df["lemmatized_text"].apply(lambda x: len(set(x.split())) / len(x.split()) if len(x.split()) > 0 else 0)
df["avg_word_length"] = df["lemmatized_text"].apply(lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0)
df_test["num_tokens"] = df_test["lemmatized_text"].apply(lambda x: len(x.split()))
df_test["unique_token_ratio"] = df_test["lemmatized_text"].apply(lambda x: len(set(x.split())) / len(x.split()) if len(x.split()) > 0 else 0)
df_test["avg_word_length"] = df_test["lemmatized_text"].apply(lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0)

In [23]:
# calculate punctuation count
import string
df["punctuation_count"] = df["lemmatized_text"].apply(lambda x: sum([1 for char in x if char in string.punctuation]))
df_test["punctuation_count"] = df_test["lemmatized_text"].apply(lambda x: sum([1 for char in x if char in string.punctuation]))

In [24]:
# Negation feature: binary if any negation word in window of previous 3 tokens
negation_words = set(["not", "no", "never", "n't", "none", "nobody", "nothing", "neither", "nowhere", "hardly", "scarcely", "barely", "doesn't", "isn't", "wasn't", "shouldn't", "wouldn't", "couldn't", "won't", "can't", "don't"])
def negation_feature(text):
    tokens = text.split()
    return 1 if any(token in negation_words for token in tokens) else 0
df["negation"] = df["lemmatized_text"].apply(negation_feature)
df_test["negation"] = df_test["lemmatized_text"].apply(negation_feature)

In [25]:
from sklearn.preprocessing import StandardScaler
additional_features = df[["sentiment_score", "intensity_marker_count", "num_tokens",
                         "unique_token_ratio", "avg_word_length", "punctuation_count",
                         "negation"] + list(emotion_lexicon_pivot.columns)]

scaler = StandardScaler()
additional_features_scaled = scaler.fit_transform(additional_features)
additional_features_test_scaled = scaler.transform(df_test[["sentiment_score", "intensity_marker_count", "num_tokens",
                                                            "unique_token_ratio", "avg_word_length", "punctuation_count",
                                                            "negation"] + list(emotion_lexicon_pivot.columns)])

In [26]:
from scipy.sparse import hstack, csr_matrix

X_train = hstack([tfidf_word, csr_matrix(additional_features_scaled)])
X_test = hstack([tfidf_word_test, csr_matrix(additional_features_test_scaled)])

In [27]:
y_train = df["ekman_emotion"]
y_test = df_test["ekman_emotion"]
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (21071, 40017), y_train shape: (21071,)
X_test shape: (914, 40017), y_test shape: (914,)


In [28]:
n_samples, n_features = X_train.shape
dual_choice = False if n_samples > n_features else False

In [29]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import RandomizedSearchCV

In [30]:
param_grid = {
    "C": [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "max_iter": [15000, 20000, 25000, 30000],
    "class_weight": [None, "balanced"],
    "loss": ["hinge", "squared_hinge"],
    "penalty": ["l2"]  # L1 penalty often causes issues with LinearSVC
}

randomized_search = RandomizedSearchCV(
    LinearSVC(random_state=42, dual=dual_choice),
    param_grid,
    cv=3,
    n_jobs=12,
    verbose=2,
    n_iter=50,
    scoring='f1_macro',
    random_state=42,
    return_train_score=True
)

randomized_search.fit(X_train, y_train)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


C:\Projects\BUAS\Y2\block-a\fae2-nlpr-group-group-9-1\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
75 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
75 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Projects\BUAS\Y2\block-a\fae2-nlpr-group-group-9-1\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Projects\BUAS\Y2\block-a\fae2-nlpr-group-group-9-1\.venv\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Pr

,estimator,LinearSVC(dua...ndom_state=42)
,param_distributions,"{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'loss': ['hinge', 'squared_hinge'], 'max_iter': [15000, 20000, ...], ...}"
,n_iter,50
,scoring,'f1_macro'
,n_jobs,12
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [37]:
print("\n" + "="*50)
print("RANDOMIZED SEARCH RESULTS")
print("="*50)
print(f"Best parameters: {randomized_search.best_params_}")
print(f"Best CV score: {randomized_search.best_score_:.4f}")
print(f"Best estimator: {randomized_search.best_estimator_}")


RANDOMIZED SEARCH RESULTS
Best parameters: {'penalty': 'l2', 'max_iter': 20000, 'loss': 'squared_hinge', 'class_weight': 'balanced', 'C': 0.1}
Best CV score: 0.4228
Best estimator: LinearSVC(C=0.1, class_weight='balanced', dual=False, max_iter=20000,
          random_state=42)


In [38]:
from sklearn.model_selection import GridSearchCV

In [39]:
param_grid = {
    'C': np.arange(0.01, 0.5, 0.025).tolist(),
    'max_iter': [20000],
    'class_weight': ["balanced"],
    'loss': ['squared_hinge'],
    'penalty': ['l2']
}

grid_search = GridSearchCV(
    LinearSVC(random_state=42, dual=dual_choice),
    param_grid,
    cv=3,
    n_jobs=12,
    verbose=2,
    scoring='f1_macro',
    return_train_score=True
)
print("Starting grid search...")
grid_search.fit(X_train, y_train)
print("\n" + "="*50)
print("GRID SEARCH RESULTS")
print("="*50)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")
print(f"Best estimator: {grid_search.best_estimator_}")

Starting grid search...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

GRID SEARCH RESULTS
Best parameters: {'C': 0.16000000000000003, 'class_weight': 'balanced', 'loss': 'squared_hinge', 'max_iter': 20000, 'penalty': 'l2'}
Best CV score: 0.4249
Best estimator: LinearSVC(C=0.16000000000000003, class_weight='balanced', dual=False,
          max_iter=20000, random_state=42)


In [40]:
best_params = grid_search.best_params_
svm_model = LinearSVC(**best_params, random_state=42, dual=dual_choice)
svm_model.fit(X_train, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,False
,tol,0.0001
,C,0.16000000000000003
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,42


In [41]:
from sklearn.metrics import classification_report
y_pred = svm_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=emotions_test, zero_division=0, digits=3))

Classification Report:
              precision    recall  f1-score   support

       anger      0.386     0.113     0.175       150
     disgust      0.103     0.068     0.082        44
        fear      0.286     0.146     0.194        41
         joy      0.828     0.236     0.367       225
     sadness      0.423     0.096     0.156       115
    surprise      0.259     0.086     0.130        81
     neutral      0.320     0.872     0.468       258

    accuracy                          0.352       914
   macro avg      0.372     0.231     0.225       914
weighted avg      0.452     0.352     0.295       914



In [42]:
# print random misclassified examples with emotion labels
import random
misclassified_indices = np.where(y_test != y_pred)[0]
random_indices = random.sample(list(misclassified_indices), min(5, len(misclassified_indices)))
for idx in random_indices:
    print(f"Text: {df_test.iloc[idx]['lemmatized_text']}")
    print(f"True label (name): {emotions_test[y_test.iloc[idx]]}, Predicted label (name): {emotions_test[y_pred[idx]]}")
    print("-"*50)

Text: I will show you later , wipe .
True label (name): anger, Predicted label (name): neutral
--------------------------------------------------
Text: I explode and it be solely in my own defense ,
True label (name): anger, Predicted label (name): neutral
--------------------------------------------------
Text: so that he would even go there with she at all .
True label (name): anger, Predicted label (name): disgust
--------------------------------------------------
Text: trypson be an asshole .
True label (name): disgust, Predicted label (name): neutral
--------------------------------------------------
Text: we be on the dragon .
True label (name): joy, Predicted label (name): neutral
--------------------------------------------------
